# 04. Scoreboard (Text Cell)

ADR-016 §6 / ADR-017 §6 검증 — 02의 DSC + 03의 train metric을 join → r 분석, polluter hold-out, default vs tuned 가중치 grid search, LLM weight generator sanity.

합격 기준:
- Pearson r(DSC, accuracy/R²) ≥ 0.4 (튜닝 set 각 dataset 별)
- Spearman ρ ≥ 0.4
- Polluter hold-out 4/5 PASS
- 모델 5/5 양의 r

회귀 트랙은 ADR-012 Degradation Index 보조 보고 추가.

---


## 0. import + 결과 csv 로드


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr

dsc_cls   = pd.read_csv('results/text_cls_dsc_sweep.csv')
dsc_reg   = pd.read_csv('results/text_reg_dsc_sweep.csv')
train_m   = pd.read_csv('results/text_train_metrics.csv')

# join key: (dataset, polluter, level, seed)
JOIN_KEYS = ['dataset', 'polluter', 'level', 'seed']


## 1. dataset별 r(DSC, metric) 측정 — 합격선 검토


In [ ]:
def compute_r_per_dataset(dsc_df, train_df, metric_col):
    j = dsc_df.merge(train_df, on=JOIN_KEYS)
    rows = []
    for ds_name in j['dataset'].unique():
        sub = j[j['dataset'] == ds_name]
        r_p, p_p = pearsonr(sub['dsc_score'], sub[metric_col])
        r_s, p_s = spearmanr(sub['dsc_score'], sub[metric_col])
        rows.append({
            'dataset': ds_name, 'n': len(sub),
            'pearson': r_p, 'p_pearson': p_p,
            'spearman': r_s, 'p_spearman': p_s,
            'pass_r040': bool(r_p >= 0.4 and r_s >= 0.4),
        })
    return pd.DataFrame(rows)

# 분류
print('=== TEXT × CLASSIFICATION ===')
print(compute_r_per_dataset(dsc_cls, train_m[train_m['task'] == 'classification'], 'accuracy'))
# 회귀
print('=== TEXT × REGRESSION ===')
print(compute_r_per_dataset(dsc_reg, train_m[train_m['task'] == 'regression'], 'r2'))


## 2. polluter hold-out (4/5 PASS)


In [ ]:
def polluter_holdout(dsc_df, train_df, metric_col):
    j = dsc_df.merge(train_df, on=JOIN_KEYS)
    polluters = j['polluter'].unique()
    rows = []
    for held in polluters:
        sub = j[j['polluter'] != held]
        for ds_name in sub['dataset'].unique():
            sd = sub[sub['dataset'] == ds_name]
            r, _ = pearsonr(sd['dsc_score'], sd[metric_col])
            rows.append({'held_out': held, 'dataset': ds_name, 'pearson': r, 'pass': r >= 0.4})
    return pd.DataFrame(rows)

# hold-out summary: dataset × polluter


## 3. 모델별 r (5/5 양의 r)


In [ ]:
def per_model_r(train_df, dsc_df, metric_col):
    j = dsc_df.merge(train_df, on=JOIN_KEYS)
    rows = []
    for m in j['model'].unique():
        sub = j[j['model'] == m]
        r, p = pearsonr(sub['dsc_score'], sub[metric_col])
        rows.append({'model': m, 'pearson': r, 'p': p, 'positive': r > 0})
    return pd.DataFrame(rows)


## 4. Default → tuned 가중치 grid search

이미지 cell의 04 노트북 패턴 미러. dead/live 메트릭 분포 진단 + grid search로 더 높은 r 달성 가능한 가중치 찾기.

ADR-015 원칙: 본 결과는 fallback 가중치의 **갭 분석** 용도. 운영·검증 정식 가중치는 Phase 4의 LLM 호출 결과 사용.


In [ ]:
# dead 메트릭 진단 — std < 0.01인 메트릭은 r에 영향 거의 없음
METRIC_KEYS = [
    'completeness_text', 'uniqueness', 'validity', 'consistency', 'outlier_ratio',
    'class_balance', 'feature_correlation', 'label_consistency',
    'feature_informativeness', 'sample_quality_text',
]
for k in METRIC_KEYS:
    if k in dsc_cls.columns:
        print(f"{k:30s} std={dsc_cls[k].std():.4f}  range=[{dsc_cls[k].min():.3f}, {dsc_cls[k].max():.3f}]")


In [ ]:
# 가중치 grid search — live 메트릭만 자유 변수, dead는 default 유지
# (이미지 cell의 score_image_v2.py 패턴)
# Scipy.optimize.minimize로 -r 최소화 → tuned 가중치 찾기

from scipy.optimize import minimize

def grid_search_weights(dsc_df, train_df, metric_col, live_keys, default_weights):
    # ... 구현 예정 — 이미지 cell의 _dev/score_image_v2.py를 텍스트로 이식
    pass


## 5. LLM weight generator sanity

Phase 4 진입 직전 sanity (튜닝 set 1개로 5회 호출 → CV 측정).


In [ ]:
from dsc_framework.llm_weight_generator import AnthropicWeightGenerator  # 미구현 시 구현 필요

# gen = AnthropicWeightGenerator(data_type='text', task='classification')
# results = [gen.generate({'schema': ...}) for _ in range(5)]
# cv = max(np.std([r.weights[k] for r in results]) / (np.mean([r.weights[k] for r in results]) + 1e-9)
#          for k in DEFAULT_WEIGHTS_TEXT.keys())
# print('weight CV:', cv, 'fallback rate:', sum(r.used_fallback for r in results) / 5)


## 6. ADR-012 Degradation Index (회귀 트랙 보조 보고)


In [ ]:
# from dsc_framework import compute_dsc_degradation
# m_deg = compute_dsc_degradation(polluted_dsc_dict, clean_dsc_dict)
# 회귀 트랙은 r(DSC, R²) absolute + r(DSC_deg, R²_deg/R²_clean) preservation 두 값 보고


---

**다음 단계**: 합격선 통과 시 Phase 4 — held-out 측정 (`plans/20260528-02-텍스트-cell-합격선-heldout-사전등록.md`).
